# Process Single Table

In [1]:
# Importation des bibliothèques nécessaires
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration pour afficher les graphiques dans le notebook
%matplotlib inline
plt.style.use('ggplot')
sns.set(style='whitegrid')

In [2]:
# Load the dataset
df = pd.read_csv('Datasets/Tabular/Time_series/dataset_time_series.csv')

# Display basic information
print("Table shape:", df.shape)
print("Column names:", df.columns.tolist())
df.head()


Table shape: (10000, 5)
Column names: ['ID', 'TimeStep', 'Feature1', 'Feature2', 'Target']


,ID,TimeStep,Feature1,Feature2,Target
0,1,0,-0.267901,0.355255,-0.183422
1,2,1,0.959884,-0.187574,-3.280214
2,3,2,0.354067,-0.154030,3.637170
3,4,3,-1.250384,0.449387,-5.965262
4,5,4,1.374300,-1.118660,0.816653


# LSTM

In [3]:
# Import necessary libraries
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


In [4]:
target_column = "Target"

X_train, X_test, y_train, y_test = train_test_split(df.drop(columns=[target_column]), df[target_column], test_size=0.2, random_state=42)

# Normalize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrame to keep column information
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

# Function to create sequences for time series forecasting
def create_sequences(X, y, seq_length):
    xs, ys = [], []
    for i in range(len(X) - seq_length):
        xs.append(X.iloc[i:i+seq_length].values)
        ys.append(y.iloc[i+seq_length])
    return np.array(xs), np.array(ys)

# Define sequence length for LSTM
seq_length = 10  # Number of time steps to look back

# Create sequences for training and test sets
X_train_seq, y_train_seq = create_sequences(X_train, y_train, seq_length)
X_test_seq, y_test_seq = create_sequences(X_test, y_test, seq_length)

# Convert data to tensors
X_train_tensor = torch.FloatTensor(X_train_seq)  # Shape: [batch_size, seq_length, features]
y_train_tensor = torch.FloatTensor(y_train_seq).reshape(-1, 1)
X_test_tensor = torch.FloatTensor(X_test_seq)
y_test_tensor = torch.FloatTensor(y_test_seq).reshape(-1, 1)

print(f"Training data shape: {X_train_tensor.shape} - X, {y_train_tensor.shape} - y")
print(f"Test data shape: {X_test_tensor.shape} - X, {y_test_tensor.shape} - y")


Training data shape: torch.Size([7990, 10, 4]) - X, torch.Size([7990, 1]) - y
Test data shape: torch.Size([1990, 10, 4]) - X, torch.Size([1990, 1]) - y


In [5]:
# Import numpy and pandas for sequence creation
import numpy as np
import pandas as pd


In [6]:
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers=2):
        super(LSTMModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        # LSTM with multiple layers
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers=num_layers, 
                          batch_first=True, dropout=0.2)
        
        # Fully connected layer
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # Initialize hidden state with zeros
        batch_size = x.size(0)
        h0 = torch.zeros(self.num_layers, batch_size, self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, batch_size, self.hidden_size).to(x.device)
        
        # Forward propagate LSTM
        lstm_out, _ = self.lstm(x, (h0, c0))  # lstm_out shape: [batch_size, seq_length, hidden_size]
        
        # Get the output from the last time step
        out = self.fc(lstm_out[:, -1, :])
        return out


In [7]:
# Initialize model, loss function and optimizer
input_size = X_train_seq.shape[2]  # Number of features
hidden_size = 64
output_size = 1
num_layers = 2
model = LSTMModel(input_size, hidden_size, output_size, num_layers)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Create mini-batches for training
from torch.utils.data import TensorDataset, DataLoader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
batch_size = 64
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)


In [8]:
# Training loop
num_epochs = 100
best_loss = float('inf')
patience = 10
counter = 0

# For tracking metrics
train_losses = []

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0
    
    for batch_X, batch_y in train_loader:
        # Forward pass
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        
        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        # Optional gradient clipping to prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        epoch_loss += loss.item()
    
    # Calculate average loss for the epoch
    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)
    
    # Early stopping
    if avg_loss < best_loss:
        best_loss = avg_loss
        counter = 0
        # Save the model
        torch.save(model.state_dict(), 'best_lstm_model.pth')
    else:
        counter += 1
    
    if counter >= patience:
        print(f'Early stopping triggered at epoch {epoch+1}')
        break
    
    if (epoch+1) % 5 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}')

# Load the best model
model.load_state_dict(torch.load('best_lstm_model.pth'))


Epoch [5/100], Loss: 10.8089


Epoch [10/100], Loss: 10.7830


Epoch [15/100], Loss: 10.6919


Epoch [20/100], Loss: 10.4704


Epoch [25/100], Loss: 10.0396


...

Epoch [85/100], Loss: 2.7148


Epoch [90/100], Loss: 2.4979


Epoch [95/100], Loss: 2.2632


Epoch [100/100], Loss: 2.1420


<All keys matched successfully>

In [9]:
# Evaluate the model
model.eval()
with torch.no_grad():
    # Make predictions
    test_predictions = model(X_test_tensor)
    test_loss = criterion(test_predictions, y_test_tensor)
    
    # Calculate metrics
    from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
    
    y_pred = test_predictions.numpy().flatten()
    y_true = y_test_tensor.numpy().flatten()
    
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    
    print(f'Test Loss: {test_loss.item():.4f}')
    print(f'MSE: {mse:.4f}')
    print(f'RMSE: {rmse:.4f}')
    print(f'MAE: {mae:.4f}')
    print(f'R² Score: {r2:.4f}')


Test Loss: 17.4456
MSE: 17.4456
RMSE: 4.1768
MAE: 3.3687
R² Score: -0.5596
